# Code to donwload Images from .parquet

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import asyncio
import aiohttp
from aiohttp import ClientTimeout
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import warnings

In [2]:

# Ignore specific PIL warning about palette transparency
warnings.filterwarnings(
    "ignore",
    "Palette images with Transparency expressed in bytes should be converted to RGBA images",
    category=UserWarning,
)

In [8]:
# ---------------- CONFIG ----------------
# DOWNLOAD_ROOT = r"E:\Thesis\0000_images"
# FAILED_LOG = r"E:\Thesis\0000_failed.log"
DOWNLOAD_ROOT = r"G:\Thesis\0000_images" #r"F:\Thesis\0001_images"
FAILED_LOG = r"G:\Thesis\0000_failed.log" #r"F:\Thesis\0001_failed.log"
BATCH_SIZE = 500
MAX_THREADS = 16 #128
# ----------------------------------------

# List parquet files explicitly
# parquet_files = [
#     r"E:\Thesis\0000_embeddings\part-00000.parquet",
#     r"E:\Thesis\0000_embeddings\part-00001.parquet",
#     r"E:\Thesis\0000_embeddings\part-00002.parquet",
#     r"E:\Thesis\0000_embeddings\part-00003.parquet",
#     r"E:\Thesis\0000_embeddings\part-00004.parquet",
#     r"E:\Thesis\0000_embeddings\part-00005.parquet",
#     r"E:\Thesis\0000_embeddings\part-00006.parquet",
#     r"E:\Thesis\0000_embeddings\part-00007.parquet",
#     r"E:\Thesis\0000_embeddings\part-00008.parquet"
# ]

# parquet_files = [
#     r"F:\Thesis\0001_embeddings\part-00000.parquet",
#     r"F:\Thesis\0001_embeddings\part-00001.parquet",
#     r"F:\Thesis\0001_embeddings\part-00002.parquet",
#     r"F:\Thesis\0001_embeddings\part-00003.parquet",
#     r"F:\Thesis\0001_embeddings\part-00004.parquet",
#     r"F:\Thesis\0001_embeddings\part-00005.parquet",
#     r"F:\Thesis\0001_embeddings\part-00006.parquet",
#     r"F:\Thesis\0001_embeddings\part-00007.parquet",
#     r"F:\Thesis\0001_embeddings\part-00008.parquet"
# ]

parquet_files = [
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00000.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00001.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00002.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00003.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00004.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00005.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00006.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00007.parquet",
    r"D:\ThesisFiles\test\0000_embeddings_cleaned\part-00008.parquet"
]



In [4]:
import socket
import time

def check_internet(host="8.8.8.8", port=53, timeout=3):
    """Check internet connectivity by attempting a TCP connection."""
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except socket.error:
        return False

def wait_for_internet(max_wait=3600):
    """
    Wait until internet connection is available.
    Exponential backoff up to 1hr total wait.
    Returns True if internet is restored, False if timeout reached.
    """
    wait_time = 5  # start with 5 seconds
    total_wait = 0
    was_offline = False

    while not check_internet():
        was_offline = True
        if total_wait >= max_wait:
            print("❌ No internet connection for 1 hour. Exiting...")
            return False
        print(f"❌ No internet connection. Retrying in {wait_time} seconds...")
        time.sleep(wait_time)
        total_wait += wait_time
        wait_time = min(wait_time * 2, 3600)  # grow wait but cap at 1hr

    if was_offline:
        print("✅ Internet connection restored!")
    return True

In [5]:
# import pandas as pd
# import pyarrow.parquet as pq
# from pathlib import Path
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import requests
# from PIL import Image
# from io import BytesIO
# from tqdm import tqdm
# import sys
# from urllib.parse import urlparse

# # ---------------- CONFIG ----------------
# # DOWNLOAD_ROOT = r"E:\Thesis\0000_images"
# # FAILED_LOG = r"E:\Thesis\0000_failed.log"
# # BATCH_SIZE = 500
# # MAX_THREADS = 50  # adjust to available resources
# # ----------------------------------------

# # Load failures from previous runs
# failed_set = set()
# failed_log_path = Path(FAILED_LOG)
# if failed_log_path.exists():
#     with open(failed_log_path, "r") as f:
#         failed_set = {line.strip() for line in f}


# def download_image(idx, url):
#     image_path = Path(DOWNLOAD_ROOT) / f"{idx}.jpg"
#     if image_path.exists():
#         return True
#     try:
#         if not wait_for_internet():
#             sys.exit("❌ Fatal Error: No internet connection for 1 hour. Job terminated.")

#         parsed = urlparse(url)
#         referer = f"{parsed.scheme}://{parsed.hostname}/"

#         headers = {
#             "User-Agent": (
#                 "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
#                 "AppleWebKit/537.36 (KHTML, like Gecko) "
#                 "Chrome/123.0.0.0 Safari/537.36"
#             ),
#             "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
#             "Accept-Language": "en-US,en;q=0.9",
#             "Referer": referer,
#             "DNT": "1",  # Do Not Track
#             "Connection": "keep-alive",
#         }

#         with requests.Session() as session:  # auto-close session
#             # resp = session.get(url, timeout=15, headers=headers)
#             resp = session.get(url, timeout=1, headers=headers) #5
#             # resp = requests.get(url, timeout=15, headers=headers)
#             resp.raise_for_status()
#             image = Image.open(BytesIO(resp.content)).convert("RGB")
#             image.save(image_path)
#             return True
#     except Exception:
#         with open(FAILED_LOG, "a") as f:
#             f.write(f"{idx}\n")
#             f.flush()
#         return False


# def process_parquet(parquet_path):
#     download_dir = Path(DOWNLOAD_ROOT)
#     download_dir.mkdir(parents=True, exist_ok=True)
#     existing_images = {p.stem for p in download_dir.glob("*.jpg") if p.stem.isdigit()}

#     parquet = pq.ParquetFile(parquet_path)
#     total_to_download = 0

#     # First count total images
#     for rg in range(parquet.num_row_groups):
#         print(f"Row Group: {rg}")
#         df = parquet.read_row_group(rg).to_pandas()
#         df = df[df["embeddings_result"].notnull()]
#         for _, row in df.iterrows():
#             idx = str(row["original_image_index"])
#             if idx not in existing_images and idx not in failed_set:
#                 total_to_download += 1

#     print(f"{parquet_path}: {total_to_download} images to download.")

#     with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
#         with tqdm(total=total_to_download, desc=Path(parquet_path).stem) as pbar:
#             futures = []
#             for rg in range(parquet.num_row_groups):
#                 df = parquet.read_row_group(rg).to_pandas()
#                 df = df[df["embeddings_result"].notnull()]

#                 batch = []
#                 for _, row in df.iterrows():
#                     idx = str(row["original_image_index"])
#                     if idx not in existing_images and idx not in failed_set:
#                         batch.append((idx, row["url"]))

#                     if len(batch) >= BATCH_SIZE:

#                         # Wait for internet before firing batch
#                         if not wait_for_internet():
#                            sys.exit("❌ Fatal Error: No internet connection for 1 hour. Job terminated.")

#                         for idx_url in batch:
#                             futures.append(executor.submit(download_image, *idx_url))
#                         batch = []

#                         # Update progress as futures complete
#                         for future in as_completed(futures):
#                             pbar.update(1)
#                         futures = []

#                 # Remaining batch
#                 if batch:
#                     if not wait_for_internet():
#                         sys.exit("❌ Fatal Error: No internet connection for 1 hour. Job terminated.")

#                     for idx_url in batch:
#                         futures.append(executor.submit(download_image, *idx_url))
#                     for future in as_completed(futures):
#                         pbar.update(1)
#                     futures = []


# def main(parquet_files):
#     for parquet_path in parquet_files:
#         print(f"Processing {parquet_path}")
#         process_parquet(parquet_path)
#     print("All downloads completed.")


In [6]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import sys
from urllib.parse import urlparse

# ---------------- CONFIG ----------------
# DOWNLOAD_ROOT = r"E:\Thesis\0000_images"
# FAILED_LOG = r"E:\Thesis\0000_failed.log"
# BATCH_SIZE = 500
# MAX_THREADS = 50  # adjust to available resources
# ----------------------------------------

# Load failures from previous runs
failed_set = set()
failed_log_path = Path(FAILED_LOG)
if failed_log_path.exists():
    with open(failed_log_path, "r") as f:
        failed_set = {line.strip() for line in f}


def download_image(idx, url):
    image_path = Path(DOWNLOAD_ROOT) / f"{idx}.jpg"
    if image_path.exists():
        return True

    try:
        if not wait_for_internet():
            sys.exit("❌ Fatal Error: No internet connection for 1 hour. Job terminated.")

        parsed = urlparse(url)
        referer = f"{parsed.scheme}://{parsed.hostname}/"

        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/123.0.0.0 Safari/537.36"
            ),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": referer,
            "DNT": "1",  # Do Not Track
            "Connection": "keep-alive",
        }

        with requests.Session() as session:
            # shorter timeout keeps threads from hanging too long on dead hosts
            resp = session.get(url, timeout=1, headers=headers)
            resp.raise_for_status()

            # Load and convert image in memory
            image = Image.open(BytesIO(resp.content)).convert("RGB")

            # Save image to an in-memory buffer first (reduces small random writes)
            buf = BytesIO()
            image.save(buf, format="JPEG", quality=90)
            img_bytes = buf.getvalue()

            # Perform a single sequential write to disk
            with open(image_path, "wb") as f:
                f.write(img_bytes)

            return True

    except Exception:
        with open(FAILED_LOG, "a") as f:
            f.write(f"{idx}\n")
            f.flush()
        return False


def process_parquet(parquet_path):
    download_dir = Path(DOWNLOAD_ROOT)
    download_dir.mkdir(parents=True, exist_ok=True)
    existing_images = {p.stem for p in download_dir.glob("*.jpg") if p.stem.isdigit()}

    parquet = pq.ParquetFile(parquet_path)
    total_to_download = 0

    # First count total images
    for rg in range(parquet.num_row_groups):
        print(f"Row Group: {rg}")
        df = parquet.read_row_group(rg).to_pandas()
        df = df[df["embeddings_result"].notnull()]
        for _, row in df.iterrows():
            idx = str(row["original_image_index"])
            if idx not in existing_images and idx not in failed_set:
                total_to_download += 1

    print(f"{parquet_path}: {total_to_download} images to download.")

    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        with tqdm(total=total_to_download, desc=Path(parquet_path).stem) as pbar:
            futures = []
            for rg in range(parquet.num_row_groups):
                df = parquet.read_row_group(rg).to_pandas()
                df = df[df["embeddings_result"].notnull()]

                batch = []
                for _, row in df.iterrows():
                    idx = str(row["original_image_index"])
                    if idx not in existing_images and idx not in failed_set:
                        batch.append((idx, row["url"]))

                    if len(batch) >= BATCH_SIZE:
                        if not wait_for_internet():
                            sys.exit("❌ Fatal Error: No internet connection for 1 hour. Job terminated.")

                        for idx_url in batch:
                            futures.append(executor.submit(download_image, *idx_url))
                        batch = []

                        for future in as_completed(futures):
                            pbar.update(1)
                        futures = []

                # Remaining batch
                if batch:
                    if not wait_for_internet():
                        sys.exit("❌ Fatal Error: No internet connection for 1 hour. Job terminated.")

                    for idx_url in batch:
                        futures.append(executor.submit(download_image, *idx_url))
                    for future in as_completed(futures):
                        pbar.update(1)
                    futures = []


def main(parquet_files):
    for parquet_path in parquet_files:
        print(f"Processing {parquet_path}")
        process_parquet(parquet_path)
    print("All downloads completed.")


In [10]:
import aiofiles
await main(parquet_files)

Processing D:\ThesisFiles\test\0000_embeddings_cleaned\part-00000.parquet
Row Group: 0
Row Group: 1
Row Group: 2
Row Group: 3
Row Group: 4
Row Group: 5
Row Group: 6
Row Group: 7
Row Group: 8
Row Group: 9
D:\ThesisFiles\test\0000_embeddings_cleaned\part-00000.parquet: 0 images to download.


part-00000: 0it [00:42, ?it/s]


Processing D:\ThesisFiles\test\0000_embeddings_cleaned\part-00001.parquet
Row Group: 0
Row Group: 1
Row Group: 2
Row Group: 3
Row Group: 4
Row Group: 5
Row Group: 6
Row Group: 7
Row Group: 8
Row Group: 9
D:\ThesisFiles\test\0000_embeddings_cleaned\part-00001.parquet: 6 images to download.


part-00001: 100%|██████████| 6/6 [00:45<00:00,  7.58s/it]


Processing D:\ThesisFiles\test\0000_embeddings_cleaned\part-00002.parquet
Row Group: 0
Row Group: 1
Row Group: 2
Row Group: 3
Row Group: 4
Row Group: 5
Row Group: 6
Row Group: 7
Row Group: 8
Row Group: 9
D:\ThesisFiles\test\0000_embeddings_cleaned\part-00002.parquet: 3 images to download.


...

part-00007: 100%|██████████| 43962/43962 [30:11<00:00, 24.27it/s]   


Processing D:\ThesisFiles\test\0000_embeddings_cleaned\part-00008.parquet
Row Group: 0
Row Group: 1
D:\ThesisFiles\test\0000_embeddings_cleaned\part-00008.parquet: 8261 images to download.


part-00008: 100%|██████████| 8261/8261 [05:55<00:00, 23.25it/s]  


All downloads completed.


TypeError: object NoneType can't be used in 'await' expression

In [3]:
import os
import json
import faiss
import heapq
import shutil
import numpy as np
from transformers import AutoProcessor, AutoModel
import torch
import pickle
from pathlib import Path
from typing import List, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


# ============================================================
# UTILITY: Chunk list
# ============================================================

def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


# ============================================================
# UTILITY: Prompt generation
# ============================================================

def generate_custom_list(object_list, template="{object}"):
    return [template.format(object=o) for o in object_list]


# ============================================================
# LOAD PROCESSED PROMPTS FROM JSONL
# ============================================================

def load_processed_prompts_from_jsonl(jsonl_path: str):
    """
    Reads JSONL file line-by-line and extracts existing prompts.
    This is lightweight and resumable.
    """
    processed = set()
    path = Path(jsonl_path)

    if not path.exists():
        return processed

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                p = obj.get("prompt")
                if p:
                    processed.add(p)
            except json.JSONDecodeError:
                print("WARNING: Skipped corrupted JSONL line.")
                continue

    print(f"[Resume] Found {len(processed)} previously processed prompts.")
    return processed


# ============================================================
# APPEND RESULTS TO JSONL
# ============================================================

def append_results_to_jsonl(jsonl_path: str, batch_results: list):
    """
    Append one line per prompt to the JSONL file.
    Atomic per-line write to avoid corruption.
    """
    path = Path(jsonl_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        for entry in batch_results:
            f.write(json.dumps(entry) + "\n")
            f.flush()
            os.fsync(f.fileno())

    print(f"[JSONL Append] {len(batch_results)} prompts appended → {jsonl_path}")


# ============================================================
# FAISS SEARCH FUNCTION (NO COPYING)
# ============================================================

def search_images(
    shard_index_paths: List[str],
    shard_mapping_paths: List[str],
    model_name: str,
    prompts: List[str],
    negative_prompts: Optional[List[str]] = None,
    top_k: int = 10,
    similarity_threshold: Optional[float] = None,
    device: str = None
):
    """
    Batch FAISS search across shards.
    Does NOT copy images or write any files.
    """

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # Load CLIP model
    processor = AutoProcessor.from_pretrained(model_name, force_download=False)
    model = AutoModel.from_pretrained(model_name, force_download=False).to(device)
    model.eval()

    # Encode prompts
    inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    # Negative prompts
    if negative_prompts:
        neg_inputs = processor(text=negative_prompts, return_tensors="pt", padding=True,
                               truncation=True).to(device)
        with torch.no_grad():
            neg_features = model.get_text_features(**neg_inputs)
        neg_features = neg_features / neg_features.norm(p=2, dim=-1, keepdim=True)
        neg_mean = neg_features.mean(dim=0, keepdim=True)
        text_features = text_features - neg_mean
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    # Convert to numpy
    query_embs = text_features.cpu().numpy().astype("float32")
    faiss.normalize_L2(query_embs)
    n_queries = query_embs.shape[0]

    # Storage
    all_results = [[] for _ in range(n_queries)]

    # SHARD SEARCH with progress bar
    num_shards = len(shard_index_paths)
    for idx_path, map_path in tqdm(
        list(zip(shard_index_paths, shard_mapping_paths)),
        desc="Searching shards",
        total=num_shards,
        unit="shard"
    ):
        idx_path = Path(idx_path)
        map_path = Path(map_path)

        shard_name = idx_path.stem
        parts = shard_name.split("_")
        group_id = parts[1] if len(parts) > 1 else "unknown"

        index = faiss.read_index(str(idx_path))
        with open(map_path, "rb") as f:
            idx_to_path = pickle.load(f)

        per_shard_k = min(len(idx_to_path), top_k)
        D, I = index.search(query_embs, per_shard_k)

        for qi in range(n_queries):
            for score, idx in zip(D[qi], I[qi]):
                if idx < 0 or idx >= len(idx_to_path):
                    continue
                if similarity_threshold is not None and score < similarity_threshold:
                    continue

                all_results[qi].append({
                    "image_path": idx_to_path[idx],
                    "score": float(score),
                    "shard": shard_name,
                    "group_id": group_id
                })

        del index

    # GLOBAL TOP-K
    final_results = []
    for qi in range(n_queries):
        final_results.append({
            "prompt": prompts[qi],
            "results": heapq.nlargest(top_k, all_results[qi], key=lambda x: x["score"])
        })

    return final_results

# ============================================================
# GLOBAL IMAGE COPY (SAFE, ATOMIC)
# ============================================================

def copy_results_global(all_results, image_dir, output_dir, max_workers=4):

    print("\n=== GLOBAL IMAGE COPY START ===")

    src_root = Path(image_dir)
    out_root = Path(output_dir)
    out_root.mkdir(parents=True, exist_ok=True)

    max_workers = max(1, min(max_workers, 8))

    def resolve_src(r):
        raw = Path(r["image_path"])
        if raw.is_absolute():
            return raw
        return src_root / raw.parent.parent / f"{r['group_id']}_images" / raw.name

    def atomic_copy(src: Path, dst: Path):
        tmp_dst = dst.with_suffix(dst.suffix + ".tmp")
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            with open(src, "rb") as fsrc, open(tmp_dst, "wb") as fdst:
                shutil.copyfileobj(fsrc, fdst, 1024 * 1024)
                fdst.flush()
                os.fsync(fdst.fileno())
            os.replace(tmp_dst, dst)
        except:
            if tmp_dst.exists():
                tmp_dst.unlink()
            raise

    # Flatten
    items = []
    for entry in all_results:
        prompt = entry["prompt"]
        pdir = out_root / prompt.replace(" ", "_")
        for r in entry["results"]:
            items.append((pdir, r))

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = []
        with tqdm(total=len(items), desc="Copying images", unit="img") as pbar:
            for pdir, r in items:
                src = resolve_src(r)
                dst = pdir / f"{r['score']:.3f}_{r['group_id']}_{src.name}"
                futures.append(ex.submit(atomic_copy, src, dst))

            for f in as_completed(futures):
                try:
                    f.result()
                except Exception as e:
                    print("Copy Error:", e)
                pbar.update(1)

    print("=== GLOBAL IMAGE COPY COMPLETE ===\n")

# ============================================================
# COPY FROM JSONL
# ============================================================

def copy_results_from_jsonl(jsonl_path, image_dir, output_dir, max_workers=4):
    all_results = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                all_results.append(obj)
            except:
                print("WARNING: Corrupt JSONL entry skipped.")

    return copy_results_global(
        all_results=all_results,
        image_dir=image_dir,
        output_dir=output_dir,
        max_workers=max_workers
    )

# ============================================================
# MAIN PROCESSING LOOP (WITH RESUME)
# ============================================================

def process_all_prompts_with_resume(
    shard_indexes,
    shard_mappings,
    profession_list,
    prompt_templates,
    jsonl_path,
    chunk_size,
    negative_prompts = ["Cartoon", "NSFW", "Sex", "Naked", "Clothing", "Object", "Sign", "Logo"],
    model_name="openai/clip-vit-large-patch14",
    similarity_threshold=0.15,
    top_k=1000
):

    processed = load_processed_prompts_from_jsonl(jsonl_path)

    # Expand profession_list → full prompts
    all_prompts = []
    for p in sorted(set(profession_list)):
        for tmpl in prompt_templates:
            all_prompts.append(tmpl.format(object=p))

    for bidx, chunk in enumerate(chunk_list(all_prompts, chunk_size)):

        # Skip prompts already processed
        to_process = [p for p in chunk if p not in processed]

        if not to_process:
            print(f"[Batch {bidx}] All prompts already processed → Skipping.")
            continue

        print(f"[Batch {bidx}] Processing {len(to_process)} prompts: {to_process}")

        batch_results = search_images(
            shard_index_paths=shard_indexes,
            shard_mapping_paths=shard_mappings,
            model_name=model_name,
            prompts=to_process,
            negative_prompts=negative_prompts,
            top_k=top_k,
            similarity_threshold=similarity_threshold
        )

        # Append
        append_results_to_jsonl(jsonl_path, batch_results)

        # Mark processed
        for r in batch_results:
            processed.add(r["prompt"])

    print("All batches processed.\n")

c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from pathlib import Path
from tqdm import tqdm

faiss_dir = r"G:\Thesis\image_retrieval_faiss_indices"

shard_indexes = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_IndexFlatIP.index")])
shard_mappings = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_mapping.pkl")])
assert len(shard_indexes) == len(shard_mappings), "Mismatch between indexes and mappings!"

# List of professions retrieved from papers as well as ChatGPT
profession_list = [
    "Accountant", "Actor", "Actuary", "Administrator", "Air Traffic Controller",
    "Airplane Pilot", "Analyst", "Animal Trainer", "Anthropologist", "Archaeologist",
    "Architect", "Artist", "Astronaut", "Athlete", "Attorney",
    "Auditor", "Baker", "Banker", "Barber", "Barista",
    "Bartender", "Biologist", "Blacksmith", "Content Creator", "Bodyguard",
    "Bounty Hunter", "Brewer", "Bricklayer", "Broker", "Builder",
    "Butcher", "Carpenter", "Caregiver", "Chef", "Chemist",
    "Chiropractor", "Civil Engineer", "Cleaner", "Clerk", "Coach",
    "Comedian", "Composer", "Construction Worker", "Therapist", "Cook",
    "Dancer", "Data Scientist", "Delivery Driver", "Dentist", "Designer",
    "Detective", "Dietitian", "Director", "DJ", "Doctor",
    "Dog Walker", "Driver", "Editor", "Electrician", "Engineer",
    "Entrepreneur", "Farmer", "Fashion Designer", "Firefighter", "Fisherman",
    "Flight Attendant", "Florist", "Gardener", "Geologist", "Graphic Designer",
    "Hairdresser", "Handyman", "Historian", "Hotel Concierge", "Ice Cream Maker",
    "Illustrator", "Translator", "Janitor", "Journalist", "Judge",
    "Laborer", "Lawyer", "Librarian", "Lifeguard", "Logger",
    "Magician", "Makeup Artist", "Marine Biologist", "Mathematician", "Mechanic",
    "Medical Researcher", "Meteorologist", "Midwife", "Miner", "Model",
    "Musician", "News Anchor", "Nurse", "Nutritionist", "Oceanographer",
    "Office Assistant", "Optician", "Painter", "Paramedic", "Park Ranger",
    "Pastry Chef", "Personal Trainer", "Pharmacist", "Photographer", "Physical Therapist",
    "Physicist", "Pilot", "Plumber", "Police Officer", "Politician",
    "Professor", "Software Engineer", "Psychologist", "Realtor", "Researcher",
    "Sailor", "Salesperson", "Scientist", "Screenwriter", "Security Officer",
    "Singer", "Skilled Technician", "Social Worker", "Soldier", "Sound Engineer",
    "Statistician", "Surgeon", "Tailor", "Teacher", "Technician",
    "Veterinarian", "Videographer", "Waiter", "Welder", "Writer",
    "Zoologist","Actuarial Analyst", "Administrative Assistant", "Appraiser", "Archivist", "Art Director",
    "Audio Technician", "Automotive Designer", "Baker Assistant", "Bankruptcy Specialist", "Bioinformatician",
    "Biomedical Engineer", "Brand Manager", "Budget Analyst", "Cartographer", "Chemical Engineer",
    "Urban Planner", "Claims Adjuster", "Clinical Laboratory Scientist", "Compliance Officer", "Conservation Officer",
    "Copywriter", "Court Reporter", "Crime Scene Investigator", "Customer Support Specialist", "Database Administrator",
    "Debt Counselor", "Economist", "Electrical Technician", "Emergency Management Specialist", "Environmental Engineer",
    "Ergonomist", "Estate Planner", "Event Coordinator", "Executive Assistant", "Facilities Manager",
    "Financial Analyst", "Flight Dispatcher", "Forensic Scientist", "Freight Coordinator", "Development Officer",
    "Genetic Counselor", "Grant Writer", "Health Inspector", "Human Resources Specialist", "Industrial Designer",
    "Insurance Underwriter", "Investment Banker", "IT Support Specialist", "Paralegal", "Loan Officer",
    "Logistics Manager", "Market Research Analyst", "Marketing Manager", "Occupational Therapist", "Operations Manager",
    "Payroll Specialist", "Procurement Officer", "Property Manager", "Quality Assurance Inspector", "Roofer"
]

prompt_templates = ["Male {object}", "Female {object}"]

jsonl_path = r"G:\Thesis\ImageRetrieval\Professions\test_retrieval_results.jsonl"

process_all_prompts_with_resume(
    shard_indexes=shard_indexes,
    shard_mappings=shard_mappings,
    profession_list=profession_list,
    prompt_templates=prompt_templates,
    jsonl_path=jsonl_path,
    chunk_size=5,
    negative_prompts = ["Cartoon", "NSFW", "Sex", "Naked", "Clothing", "Object", "Sign", "Logo"],
    model_name="openai/clip-vit-large-patch14",
    similarity_threshold=0.15,
    top_k=1_000#10_000
)

# copy_results_from_jsonl(
#     jsonl_path=jsonl_path,
#     image_dir=r"G:\Thesis",
#     output_dir=r"G:\Thesis\ImageRetrieval\Professions",
#     max_workers=4
# )

[Batch 0] Processing 4 prompts: ['Male Accountant', 'Female Accountant', 'Male Actor', 'Female Actor']


c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Searching shards:  33%|███▎      | 12/36 [07:25<29:04, 72.69s/shard]